### Tools

Models can request to call tools that perforn tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions(often a JSON schema)
2. A function or coroutine to execute.

In [2]:
import os
from langchain.chat_models import init_chat_model
os.environ['GOOGLE_API_KEY'] = os.getenv("GOOGLE_API_KEY")
model = init_chat_model("google_genai:gemini-3-flash-preview")

response=model.invoke("Write a essay on AI")
response

AIMessage(content=[{'type': 'text', 'text': '**Title:** The Double-Edged Sword: The Evolution and Impact of Artificial Intelligence\n\n**Introduction**\nIn the mid-20th century, the concept of a machine that could "think" was relegated to the realms of science fiction. Today, Artificial Intelligence (AI) is no longer a futuristic fantasy; it is an invisible thread woven into the fabric of daily life. From the algorithms that curate our social media feeds to the diagnostic tools helping doctors identify diseases, AI is fundamentally reshaping how we live, work, and interact. While AI promises a new era of unprecedented efficiency and innovation, it also presents profound ethical and societal challenges that require careful navigation.\n\n**The Evolution of AI**\nThe journey of AI began with the ambition to replicate human cognition. Early pioneers like Alan Turing questioned whether machines could imitate human intelligence, leading to the development of simple rule-based systems. Howev

# Tools

In [3]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"Its sunny in {location}"

model_with_tools = model.bind_tools([get_weather])


In [4]:
response = model_with_tools.invoke("Whats the weather in banglore")
print(response)

content=[] additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Bangalore"}'}, '__gemini_function_call_thought_signatures__': {'67d94333-10f2-409c-bde1-554b0d8282ac': 'EngKdgEMOdbHwinUwtX/x5TMX+z1yw4O1DARi6L9Fc+Bpuns2MjKWmqabpFiZeAqz32sQgS21ZO6QI6XKnJRmdrhmk4A3Q91Sfk1Dis8OTi/L0xc3lnCOKMaqfZzJvUmqcRSMX24KNevVuPutW586vLKsdSCvuwqj04='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019e3b64-1150-73b3-af9e-a13ed27547ca-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Bangalore'}, 'id': '67d94333-10f2-409c-bde1-554b0d8282ac', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 50, 'output_tokens': 30, 'total_tokens': 80, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 13}}


In [5]:
response.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Bangalore'},
  'id': '67d94333-10f2-409c-bde1-554b0d8282ac',
  'type': 'tool_call'}]

### Tool execution loop

In [6]:
# step 1: Model generates tool calls
messages = [{"role": "user", "content": "Whats the weather in Banglore"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)
    
final_response = model_with_tools.invoke(messages)
print(final_response.text)

It's currently sunny in Bangalore.


In [7]:
messages

[{'role': 'user', 'content': 'Whats the weather in Banglore'},
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Bangalore"}'}, '__gemini_function_call_thought_signatures__': {'93189fdd-42df-47ba-b21d-f61772383571': 'EqcBCqQBAQw51sdkEA5cnbQM5qItQSkGMgNAPeN4L4VsMoRQrUA0pjTvo3wtkBaKc4uoBz1kR5qKphnPYe1ZWgIDmsFO3CYTzgl6n9b9ZA4M6eHwiEke3HmqGwO1C9kpXNG2pzYeXc7pOfdEyiM8gVOpfQjfwJXHueAlpHq6qIlVSP55kemQxscQYltg1G7bFe/ejjLLRN3TWJC3nbuHpt1IaYABrr3FGTA='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e3b64-16c9-7d63-a323-508009935119-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Bangalore'}, 'id': '93189fdd-42df-47ba-b21d-f61772383571', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 50, 'output_tokens': 42, 'total_tokens': 92, 'input_token_details': {'cache_read': 0}, 'output_token

In [8]:
model_with_tools

_ChatModelBinding(bound=ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 3 Flash Preview', 'release_date': '2025-12-17', 'last_updated': '2025-12-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-3-flash-preview', temperature=1.0, client=<google.genai.client.Client object at 0x000002227131CA10>, default_metadata=(), model_kwargs={}), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_weather', 'description': 'Get the weather at a location', 'parameters': {'properti